<img src="https://datascientest.fr/train/assets/logo_datascientest.png" style="height:100px" alt="MCUNet Logo">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> Déploiement TinyML avec MCUNet </h1></center>
<center><h2> Notebook 01 : Exploration du Model Zoo (Solution) </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

>
> **Objectif :** Comprendre la structure du Model Zoo MCUNet et choisir un modèle adapté à un MCU cible en fonction de ses contraintes de mémoire.
>
> **Durée estimée :** 30 minutes
>
> **Prérequis :** Avoir validé l'environnement (Notebook 00).
>


### Concepts clés
>
> 1. **SRAM (Static RAM)** : Mémoire volatile très rapide mais coûteuse et rare. C'est ici que sont stockées les **activations** (les features maps intermédiaires) pendant l'inférence. Le goulot d'étranglement principal en TinyML.
> 2. **Flash Memory** : Mémoire non volatile où est stocké le programme et les **poids du modèle** (en lecture seule).
> 3. **MACs (Multiply-Accumulate Operations)** : Mesure de la complexité de calcul. 1 MAC = 1 multiplication + 1 addition.


--- 
### 1. Définition de la cible Hardware

Notre cible est un **STM32F746**. C'est un microcontrôleur basé sur une architecture ARM Cortex-M7 tournant à 216 MHz.

Voici ses spécifications brutes :
- **SRAM totale :** 320 KB
- **Flash totale :** 1 MB (1024 KB)

Cependant, le modèle de deep learning ne peut pas utiliser 100% de cette mémoire. Il faut laisser de la place pour le firmware (le code C/C++ de l'application), le RTOS (système d'exploitation temps réel), et les buffers d'entrée/sortie.

In [ ]:
# Spécifications brutes
TOTAL_SRAM_KB = 320
TOTAL_FLASH_KB = 1024

# Réserves pour le système (estimations)
OS_RESERVE_SRAM_KB = 50
FW_RESERVE_FLASH_KB = 250

# Budget net disponible pour le modèle ML
budget_sram = TOTAL_SRAM_KB - OS_RESERVE_SRAM_KB
budget_flash = TOTAL_FLASH_KB - FW_RESERVE_FLASH_KB

print(f"--- Budget Mémoire STM32F746 ---")
print(f"SRAM disponible pour ML  : {budget_sram} KB")
print(f"Flash disponible pour ML : {budget_flash} KB")

--- 
### 2. Explorer les modèles MCUNet (Tâche VWW)

Les auteurs de MCUNet fournissent un "Model Zoo" de modèles pré-entraînés avec différentes architectures (TinyNAS) optimisées pour différents budgets de mémoire. Nous allons extraire les statistiques des modèles VWW (Visual Wake Words).

In [ ]:
import pandas as pd

# Données issues du README officiel de MCUNet (int8 quantization)
vww_models_data = [
    {"net_id": "mcunet-vww0", "macs_m": 6.0, "params_m": 0.37, "sram_kb": 146, "flash_kb": 617, "resolution": 64, "acc_top1": 87.3},
    {"net_id": "mcunet-vww1", "macs_m": 11.6, "params_m": 0.43, "sram_kb": 162, "flash_kb": 689, "resolution": 80, "acc_top1": 88.9},
    {"net_id": "mcunet-vww2", "macs_m": 55.8, "params_m": 0.64, "sram_kb": 311, "flash_kb": 897, "resolution": 144, "acc_top1": 91.8}
]

df_vww = pd.DataFrame(vww_models_data)
display(df_vww)

--- 
### 3. Petit exercice pratique

Écrire une fonction `select_best_model` qui prend en entrée le budget SRAM, le budget Flash et une précision minimale requise, et qui retourne le `net_id` du meilleur modèle compatible.

Le "meilleur" modèle est celui qui respecte les contraintes et qui offre la plus grande précision (`acc_top1`).

In [ ]:
def select_best_model(models_df, max_sram, max_flash, min_acc):
    valid_models = models_df[
        (models_df['sram_kb'] <= max_sram) &
        (models_df['flash_kb'] <= max_flash) &
        (models_df['acc_top1'] >= min_acc)
    ]
    
    if valid_models.empty:
        return None
        
    sorted_models = valid_models.sort_values(by='acc_top1', ascending=False)
    return sorted_models.iloc[0]['net_id']

# Testons la fonction avec notre budget calculé précédemment
best_model_id = select_best_model(df_vww, budget_sram, budget_flash, min_acc=85.0)
print(f"Le meilleur modèle pour STM32F746 est : {best_model_id}")

<div class="alert alert-info">
<i class="fa fa-info-circle"></i> &emsp; 
Le modèle <code>mcunet-vww2</code> a une excellente précision, mais avez-vous remarqué pourquoi il n'est pas sélectionné ? 
Il a besoin de 311 KB de SRAM, alors que notre budget disponible n'est que de 270 KB.
</div>

--- 
### Points d'attention

<div class="alert alert-warning">
<i class='fa fa-exclamation-triangle '></i> &emsp; 
  <strong>La "taille" d'un modèle n'est pas qu'une question de paramètres.</strong> Les développeurs ML classiques regardent souvent le nombre de paramètres pour juger de la taille d'un réseau. En TinyML, ce sont souvent les <strong>activations (SRAM)</strong> qui bloquent le déploiement en premier. Un modèle avec peu de poids mais qui utilise de grandes features maps à haute résolution ne rentrera pas.
</div>

--- 
### Questions de réflexion

1. Pourquoi le modèle `mcunet-vww2` (311 KB) requiert-il près de deux fois plus de SRAM que le modèle `mcunet-vww1` (162 KB) alors que son nombre de paramètres (poids) n'est supérieur que de 50% ? 

--> Il faut regarder la colonne resolution.

2. Entre la contrainte SRAM et la contrainte Flash, laquelle est généralement la plus restrictive (le goulot d'étranglement le plus sévère) en TinyML selon les données de ce tableau et vos observations ?

--> vww2 sature la SRAM à 97% alors que la Flash a encore 12% de marge. C'est la SRAM qui détermine la faisabilité, pas la Flash. Mathématiquement, c'est également justifiable : quand on monte en gamme dans un Model Zoo, la résolution d'entrée augmente (64 -> 80 -> 144). Les activations grandissent quadratiquement avec la résolution, alors que les poids grandissent linéairement avec la profondeur du réseau. La SRAM explose plus vite que la Flash.